# Code_5b — Published Baseline: SelfCheckGPT-style Sampling Consistency

**GPU: yes** (T4). Run after Code_5. Does **not** change any Code_5 result.

**Baseline:** SelfCheckGPT (Manakul, Liusie & Gales, EMNLP 2023) — a reference-free, black-box
hallucination detector. Idea: sample the same model several times; if the samples disagree with the
original answer, the answer is probably hallucinated.

**Adaptation to tool calls** (the original works on sentences):
* the judged call = the model's original (greedy) call from Code_5;
* `N_SAMPLES` extra calls are sampled from the same model with the same prompt (temperature 1.0);
* similarity of the judged call to one sample = 0.5 × [same function] + 0.5 × Jaccard overlap of the
  (parameter, value) pairs (null / empty values = not set, as everywhere else); an unparseable sample counts as 0;
* hallucination score = 1 − mean similarity over the samples (higher = more likely wrong).

The flag threshold is chosen on a 30 % tuning part of the queries (balanced accuracy) and all
metrics are reported on the other 70 %, on exactly the same calls as the schema-only system.
AUROC is threshold-free. Labels: the automatic reference labels from Code_5; Code_6 also compares
against the human labels.

Outputs: `runs/<run_id>/selfcheck_results.json`; samples cached in `cache/selfcheck/` (resumable).

In [ ]:
import os, sys, json, importlib, subprocess, time, random
from collections import Counter, defaultdict

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hallucination_project_v2'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'accelerate'], check=True)
else:
    DRIVE_ROOT = os.environ.get('TOOLHALL_ROOT', os.path.abspath('hallucination_project_v2'))
CODE_DIR = os.path.join(DRIVE_ROOT, 'code')
sys.path.insert(0, CODE_DIR)
import toolhall_core as th
importlib.reload(th)
import numpy as np

NOTEBOOK = 'Code_5b'
CFG = th.load_config(os.path.join(CODE_DIR, 'config.json'))
CFG['drive_root'] = DRIVE_ROOT
P = th.Paths(CFG)
SEED = CFG['seed']
th.set_seed(SEED)
NB, ALPHA = CFG['bootstrap']['n'], CFG['bootstrap']['alpha']

N_SAMPLES = 3
MODELS = ['Qwen/Qwen2.5-1.5B-Instruct', 'Qwen/Qwen2.5-3B-Instruct', 'microsoft/Phi-3.5-mini-instruct']
# to run all five models: MODELS = CFG['models']['generation_models']
TEMPERATURE = 1.0
TUNE_FRAC = 0.30
QUICK_TEST = None

SC_DIR = P.cache_file('selfcheck')
os.makedirs(SC_DIR, exist_ok=True)
raw_registry = th.load_json(P.data_file('registry.json'), CFG, kind='data')
real = th.load_json(P.data_file('real_eval.json'), CFG, kind='data')
real_records = th.load_json(P.run_file('real_records.json'), CFG)
REG = th.Registry(raw_registry, CFG['validator']['unknown_required_policy'])
EVAL = [r for r in real if r['role'] == 'eval']
if QUICK_TEST:
    EVAL = EVAL[:QUICK_TEST]
EVAL_IDS = {r['id'] for r in EVAL}
ORIG = {(r['model'], r['id']): r for r in real_records
        if r['mode'] == 'free' and r['condition'] == 'pipeline' and r['id'] in EVAL_IDS}
MODELS = [m for m in MODELS if any(k[0] == m for k in ORIG)]
print(f"models: {MODELS} | queries: {len(EVAL)} | samples per query: {N_SAMPLES}")

## Same prompt as Code_5

In [ ]:
SYSTEM_PROMPT = ('You are a function-calling assistant. Choose ONE of the available functions and reply with '
                 'exactly one JSON object of the form {"function": "<name>", "parameters": {...}} and nothing else.')


def messages_for(r):
    tools = [th.schema_for_prompt(REG, f) for f in r['tool_functions'] if REG.resolve(f)]
    return [{'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f"Available functions:\n{json.dumps(tools, ensure_ascii=False)[:6000]}\n\n"
                                        f"Request: {r['query']}"}]

## Sample (resumable, cached per model)

In [ ]:
import torch
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')


def sample_path(model):
    return os.path.join(SC_DIR, f"{model.replace('/', '_')}__k{N_SAMPLES}_t{TEMPERATURE}.jsonl")


def sample_batch(tok, model, device, prompts, k, max_new_tokens):
    enc = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=3072,
              add_special_tokens=False).to(device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=TEMPERATURE,
                             top_p=1.0, num_return_sequences=k, pad_token_id=tok.pad_token_id)
    n_in = enc['input_ids'].shape[1]
    texts = [tok.decode(row[n_in:], skip_special_tokens=True).strip() for row in out]
    return [texts[i * k:(i + 1) * k] for i in range(len(prompts))]


def sample_safe(tok, model, device, prompts, k, max_new_tokens):
    """Same output as sample_batch; on GPU out-of-memory retries one prompt at a time,
    then one sample at a time, so large models fit on a T4."""
    try:
        return sample_batch(tok, model, device, prompts, k, max_new_tokens)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
    groups = []
    for p in prompts:
        try:
            groups += sample_batch(tok, model, device, [p], k, max_new_tokens)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            one = []
            for _ in range(k):
                one += sample_batch(tok, model, device, [p], 1, max_new_tokens)[0]
                torch.cuda.empty_cache()
            groups.append(one)
    return groups


for model_name in MODELS:
    done = {x['id'] for x in th.read_jsonl(sample_path(model_name))}
    todo = [r for r in EVAL if r['id'] not in done]
    if not todo:
        print(f"{model_name}: cached")
        continue
    print(f"{model_name}: {len(todo)} queries to sample")
    tok, model, device = th.load_causal_lm(model_name)
    torch.manual_seed(SEED)
    t0 = time.time()
    for start in range(0, len(todo), 4):
        chunk = todo[start:start + 4]
        prompts = [th.chat_text(tok, messages_for(r)) for r in chunk]
        groups = sample_safe(tok, model, device, prompts, N_SAMPLES, CFG['models']['max_new_tokens'])
        for r, g in zip(chunk, groups):
            th.append_jsonl(sample_path(model_name), {'id': r['id'], 'samples': g})
        if (start // 4 + 1) % 10 == 0:
            print(f"   {start + len(chunk)}/{len(todo)}  ({time.time() - t0:.0f}s)")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Consistency score and detection

In [ ]:
def pairs_of(call):
    n = th.normalize_tool_call(call)
    if n is None:
        return None, set()
    return n['function'], {(k, th.canonical_json(v)) for k, v in n['parameters'].items() if v not in (None, '')}


def similarity(judged, sample_text):
    s = th.parse_raw_call(sample_text)
    if s is None:
        return 0.0
    f1, p1 = pairs_of(judged)
    f2, p2 = pairs_of(s)
    if f1 is None or f2 is None:
        return 0.0
    jac = len(p1 & p2) / len(p1 | p2) if (p1 | p2) else 1.0
    return 0.5 * float(f1 == f2) + 0.5 * jac


def auroc(y, s):
    pos = [v for v, t in zip(s, y) if t]
    neg = [v for v, t in zip(s, y) if not t]
    if not pos or not neg:
        return None
    wins = sum((p > n) + 0.5 * (p == n) for p in pos for n in neg)
    return wins / (len(pos) * len(neg))


ids_sorted = sorted(EVAL_IDS)
random.Random(SEED).shuffle(ids_sorted)
n_tune = int(round(TUNE_FRAC * len(ids_sorted)))
TUNE_IDS, TEST_IDS = set(ids_sorted[:n_tune]), set(ids_sorted[n_tune:])

PER_RECORD, SC = {}, {}
for model_name in MODELS:
    samples = {x['id']: x['samples'] for x in th.read_jsonl(sample_path(model_name))}
    recs = []
    for rid in sorted(EVAL_IDS):
        o = ORIG.get((model_name, rid))
        if o is None or rid not in samples:
            continue
        judged = th.parse_raw_call(o['raw_output'])
        sims = [similarity(judged, s) for s in samples[rid]] if judged is not None else [0.0] * len(samples[rid])
        score = 1.0 - float(np.mean(sims))
        recs.append({'id': rid, 'score': score, 'y': int(o['is_hallucinated']), 'ours': int(o['flagged']),
                     'tool_group': o['tool_group']})
    tune = [r for r in recs if r['id'] in TUNE_IDS]
    test = [r for r in recs if r['id'] in TEST_IDS]
    best_t, best_b = 0.5, -1
    for t in sorted({r['score'] for r in tune}) + [1.01]:
        b = th.binary_metrics([r['y'] for r in tune], [int(r['score'] >= t) for r in tune])['balanced_accuracy']
        if b > best_b:
            best_t, best_b = t, b
    for r in recs:
        r['selfcheck'] = int(r['score'] >= best_t)
        PER_RECORD[f"{model_name}|{r['id']}"] = {'score': r['score'], 'flag': r['selfcheck']}
    y = [r['y'] for r in test]
    sc = [r['selfcheck'] for r in test]
    ours = [r['ours'] for r in test]
    union = [int(a or b) for a, b in zip(sc, ours)]
    caught_by_sc_only = sum(1 for r in test if r['y'] and r['selfcheck'] and not r['ours'])
    SC[model_name] = {
        'threshold': best_t, 'n_tune': len(tune), 'n_test': len(test),
        'auroc_all': auroc([r['y'] for r in recs], [r['score'] for r in recs]),
        'selfcheck': th.binary_metrics(y, sc),
        'schema_only': th.binary_metrics(y, ours),
        'union': th.binary_metrics(y, union),
        'errors_caught_only_by_selfcheck': caught_by_sc_only,
        'mcnemar_detection_correct': th.mcnemar_exact([int(a == b) for a, b in zip(ours, y)],
                                                      [int(a == b) for a, b in zip(sc, y)]),
        'ci_balanced_accuracy_selfcheck': th.bootstrap_ci(lambda a, b: th.binary_metrics(a, b)['balanced_accuracy'],
                                                          y, sc, n=NB, alpha=ALPHA, seed=SEED)}

th.print_header(f"SELFCHECK vs SCHEMA-ONLY — detection on the 70 % test queries (auto labels)")
print(f"{'model':<34}{'method':<14}{'bal.acc':>8}{'prec':>7}{'rec':>7}{'F1':>7}{'FPR':>7}")
for m, d in SC.items():
    for name in ('selfcheck', 'schema_only', 'union'):
        b = d[name]
        print(f"{m[-34:]:<34}{name:<14}{b['balanced_accuracy']:>8.3f}{b['precision']:>7.3f}{b['recall']:>7.3f}"
              f"{b['f1']:>7.3f}{b['fpr']:>7.3f}")
    print(f"{'':<34}AUROC(selfcheck) {d['auroc_all']:.3f} | threshold {d['threshold']:.3f} | McNemar p "
          f"{d['mcnemar_detection_correct']['p_value']:.3g} | errors caught only by selfcheck: "
          f"{d['errors_caught_only_by_selfcheck']}\n")

## Save

In [ ]:
th.save_json({'method': 'SelfCheckGPT-style sampling consistency adapted to tool calls (Manakul et al., EMNLP 2023)',
              'n_samples': N_SAMPLES, 'temperature': TEMPERATURE, 'models': MODELS,
              'similarity': '0.5*[same function] + 0.5*Jaccard((param, value) pairs)',
              'threshold_rule': f'balanced accuracy on a {TUNE_FRAC:.0%} tuning part of the queries',
              'tune_ids': sorted(TUNE_IDS), 'test_ids': sorted(TEST_IDS),
              'by_model': SC, 'per_record': PER_RECORD,
              'hardware': th.hardware_info(), 'packages': th.package_versions()},
             P.run_file('selfcheck_results.json'), CFG, NOTEBOOK)
print('saved ->', P.run_file('selfcheck_results.json'))
print('Code_5b complete.')